In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    TimestampType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

landing_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/carriers/"
)

bronze_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/carriers/"
)

checkpoint_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/bronze_carriers/"
)

schema_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/schemas/bronze_carriers/"
)

# ============================================================
# 2. SCHÉMA SOURCE
# ============================================================

carriers_schema = StructType([
    StructField("carrier_id", StringType(), True),
    StructField("carrier_name", StringType(), True),
    StructField("carrier_type", StringType(), True),
    StructField("country", StringType(), True),
    StructField("service_level", StringType(), True),
    StructField("active", BooleanType(), True),
    StructField("updated_at", TimestampType(), True)
])

# ============================================================
# 3. LECTURE INCRÉMENTALE AVEC AUTO LOADER
# ============================================================

carriers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_carriers_path)
    .option("header", "true")
    .schema(carriers_schema)
    .load(landing_carriers_path)
)

# ============================================================
# 4. AJOUT DES MÉTADONNÉES BRONZE
# ============================================================

bronze_carriers_df = (
    carriers_stream
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "batch_id",
        F.lit("reference_initial")
    )
)

# ============================================================
# 5. ÉCRITURE DELTA
# ============================================================

query = (
    bronze_carriers_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_carriers_path
    )
    .option(
        "path",
        bronze_carriers_path
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Ingestion Bronze carriers terminée.")

# ============================================================
# 6. LECTURE DE LA BRONZE
# ============================================================

bronze_carriers = (
    spark.read
    .format("delta")
    .load(bronze_carriers_path)
)

print(
    "Nombre de lignes Bronze carriers :",
    bronze_carriers.count()
)

display(
    bronze_carriers
    .orderBy("carrier_id")
)

# ============================================================
# 7. VÉRIFICATION DES MÉTADONNÉES
# ============================================================

display(
    bronze_carriers.select(
        "carrier_id",
        "carrier_name",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )
    .orderBy("carrier_id")
)